# Day 2 · Module 3: Sandboxing

**Objective:** understand sandboxing as the boundary on what agent-produced code can do at execution time, distinct from permissions, which bound what the agent itself may do. Run ContosoBank's real sandbox against three planted attacks and record what actually happened.


## Relevant Claude Code Docs
Review these before starting the module:
- [Choose a sandbox environment](https://code.claude.com/docs/en/sandbox-environments)
- [Configure the sandboxed Bash tool](https://code.claude.com/docs/en/sandboxing)
- [Security](https://code.claude.com/docs/en/security)

## 1 · The idea

Permissions bound what the agent may *do*: which tools it can call, which paths it can touch, which shell commands are denied. Sandboxing is a different layer entirely — it bounds what *code the agent produced* can do once something else executes it. The agent might never call a disallowed tool and still hand off a test file, a script, or a dependency that does damage the moment it runs.

A sandbox for agent-produced code execution is usually described along four boundaries:

- **Network** — can the running code reach the outside world at all?
- **Filesystem** — can it read or write anything beyond its own worktree?
- **Secrets** — can it see credentials, host environment variables, or files outside the mount?
- **Dependencies** — can it install or pull in arbitrary packages at run time?

Isolation tightens along a spectrum: a hardened container (namespaces + cgroups, still sharing a kernel) is the cheapest and most common tier; gVisor interposes a user-space kernel between the container and the host for a much smaller syscall attack surface; a microVM (e.g. Firecracker) gives each run its own lightweight virtual machine and kernel, the strongest isolation at the highest operational cost. Most teams start at the container tier and move right only when the threat model demands it.


### Grounding

Related concept: the indirect bypass from permission modeling — an implementer with no Bash access writes a test file that calls `os.system`, and the tester, legitimately running `uv run pytest`, executes it anyway. No permission rule stops that, because the tester's own action was allowed; what escaped the rule was the *content* of the code it ran. A sandbox is exactly the layer that closes that gap: it doesn't care who wrote the code or what tools they used to write it, only what the code is allowed to do once it executes.

ContosoBank's `sandbox/run-sandbox.sh` implements this for the test suite. It runs `pytest` inside a container under `--network none` (no egress), a read-only filesystem plus a scratch tmpfs (no persistent writes), only the worktree mounted (nothing else reachable), and no host environment variables inherited. Read `sandbox/README.md` and `sandbox/Dockerfile` before you start.


### Setup check

Run the cell below once per session. It makes sure `contoso` is importable and prints the resolved project root — you'll need `proj` later for the self-check.


In [ ]:
import subprocess, sys, os
# Ensure src is importable and the sandbox is healthy before you start.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1","day2") else None
root = subprocess.run(["git","rev-parse","--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))
print("project root:", proj)
try:
    import contoso
    print("contoso imported OK")
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in the repo root, then restart the kernel.")


### Predict, before you run anything

Read `reference/day2/m3/attack-tests.md` now, but do **not** run the sandbox yet. For each planted attack, write down which of the four boundaries (network / filesystem / secrets / dependencies) you expect to stop it, and what you expect the failure to look like:

| Attack | Boundary you predict stops it | Expected failure signature |
|---|---|---|
| (a) POST an API key / ledger batch to an external URL | | |
| (b) read `sandbox/planted_secret.txt` + dump `os.environ` | | |
| (c) write to an existing read-only directory (e.g. `/etc/x`) | | |

Only after you've written a prediction for all three should you move to the exercise below and compare what actually happened against what you expected. Getting one wrong is the point — it usually means you were reasoning about permissions (what the agent may do) instead of sandboxing (what the code can do once it runs).


## 2 · Your exercise

**Primary path (requires Docker):**

1. Run `./sandbox/run-sandbox.sh` once against the legitimate suite to confirm the sandbox itself builds and runs clean. Verify green by the run's own reported count of collected/passed tests — don't assume a fixed number, the suite's size drifts as code shrinks.
2. Per `reference/day2/m3/attack-tests.md`, have your implementer produce a test file with three planted attacks:
   - **(a) Network egress** — attempt to POST an account's API key or a batch of ledger entries to an external URL such as `http://collector.evil.example`.
   - **(b) Secret read** — open and read `sandbox/planted_secret.txt`, and separately dump the full contents of `os.environ`. (Don't substitute `~/.ssh` or a `.env` file — neither exists in this image, so probing them fails with "no such file" for the wrong reason and never exercises the Secrets boundary.)
   - **(c) Filesystem write outside `/work`** — attempt to write a file inside an *existing* read-only directory, e.g. `/etc/contoso-escape-test.txt` (not `/tmp` and not `/work/artifacts`, both of which are intentionally writable, and not a made-up path like `/opt/evil/x` — a missing parent directory fails with "no such file or directory" before the write ever reaches the read-only-filesystem check).
3. Run `./sandbox/run-sandbox.sh` again with the attack file present. For each attack, capture the exact error text (or the evidence of success) — do not write "blocked" without the output that shows it.
4. Confirm the pre-existing legitimate suite is still green in the same sandboxed run — again, by comparing the run's own test count, not a number you remember from a prior run.
5. Record all four outcomes in `artifacts/day2/m3/sandbox-verification.md`.

**Fallback (no Docker):** fill in `scenarios/sandbox-spec-template.md` for ContosoBank, then self-attack your own spec — list three concrete ways an agent-produced payload could defeat it, and patch the spec to close each one.


### What good looks like

All three attacks are blocked, the legitimate suite stays green (by its own reported count, not a number you hardcoded), and the write-outside-the-worktree attempt targets a real *existing* read-only path like `/etc` — not `/tmp` (intentionally writable via tmpfs), not `/work/artifacts` (intentionally mounted read-write), and not a made-up path like `/opt/evil/x`. Each outcome in your write-up is backed by the actual error text or output, not an assumption.

**The trap to name explicitly: `ENOENT` vs `EROFS`.** Writing to a path whose parent directory doesn't exist (`/opt/evil/x`) fails with `ENOENT` — "No such file or directory" — which is true regardless of whether the filesystem is read-only. That failure proves nothing about the sandbox. Writing to a path inside a directory that genuinely exists on the read-only root (`/etc/contoso-escape-test.txt`) fails with `EROFS` — "Read-only file system" — which is the actual boundary you set out to verify. Only the second one is evidence the filesystem boundary holds; the first is a wrong-reason pass that would look identical whether the sandbox worked or not.


### Verify your work

This checklist is advisory, not a gate — it just reflects back what it finds in `artifacts/day2/m3/`. Run it any time; it's safe to run before you've written anything. If Docker isn't available, this will correctly show everything missing — that's expected on the fallback path, where the deliverable lives in `scenarios/sandbox-spec-template.md` instead.


In [ ]:
import pathlib
sv = pathlib.Path(proj) / "artifacts" / "day2" / "m3" / "sandbox-verification.md"
if sv.exists():
    t = sv.read_text().lower()
    print(f"[x] sandbox-verification.md present ({len(t.split())} words)")
    print(f"  [{'x' if 'network' in t or 'egress' in t or 'http' in t else ' '}] covers the network boundary?")
    print(f"  [{'x' if 'secret' in t or 'env' in t or 'planted' in t else ' '}] covers the secrets boundary?")
    print(f"  [{'x' if '/opt' in t or 'outside' in t or '/etc' in t else ' '}] covers a write outside /work (not /tmp)?")
    print(f"  [{'x' if 'block' in t else ' '}] states what was blocked?")
else:
    print("[ ] artifacts/day2/m3/sandbox-verification.md missing")
    print("    (no-Docker path: fill scenarios/sandbox-spec-template.md instead)")


## 3 · Debrief

- Which boundary stopped which attack? Network egress, filesystem writes, and secret reads are each closed by a different mechanism — did any of them surprise you?
- When is configuration (permission rules, deny lists) enough, and when do you actually need isolation — a separate process, container, or VM — because the thing you're bounding is code you didn't write and can't fully inspect in advance?


---
### Take-home

Permissions and sandboxing answer different questions: "what may the agent do" versus "what can the code it produced do once it runs." Both are necessary and neither substitutes for the other — the indirect bypass that survives permission modeling is exactly what the sandbox is built to close.

(Related concept: bounding the workflow itself — how many agents, how much depth, how many revision loops — is a separate axis of control.)
